# Where We Left Off

In the previous lab we built a safety governor to safely move our hero asking for a set of desired joint angles, expressed in radiants, and we used forward kinematics to track the end effector position in real-time.

# In This Lab

In this lab we will do the opposite: we will move our hero directly asking for a given end effector position. Joint angles will be computed using Inverse Kinematics.

# Inverse Kinematics

Let's retrieve the 2D model developed in the first lab. 

We want to find the joint angles $\vartheta_{1}$, $\vartheta_{2}$, $\vartheta_{3}$ given the position of the end effector $P=(x_p,z_p)$. The link lengths $l_1$, $l_2$, $l_3$ are known, since we can measure them.

![2d_model_vs_arm](../lab_1_forward_kinematics/images/2d_model_vs_so-101_arm.png)

Without doing any math, we can immediately see that the problem can have **no solution** and **multiple solutions**.

## No solution

A 2D configuration where the desired end-effector position $\underline{P}_{\text{des}}$ is **out of reach**, i.e., $\|\underline{P}_{\text{des}}\|_2 > \ell_1 + \ell_2 + \ell_3$.

<img src="images/out_of_reach.png" width="600" style="display: block; margin: 0 auto;">

## Multiple solutions

Two distinct 2D configurations reaching the same end-effector pose at $\underline{P}$: elbow-down (left) and elbow-up (right). 

<div style="display: flex; justify-content: center; gap: 20px;">
  <img src="../../lab_1_forward_kinematics/images/2d_model_so-101_arm.png" style="width: 48%; max-width: 600px;">
  <img src="images/elbow_up.png" style="width: 48%; max-width: 600px;">
</div>

## Analytical Solution

We want to rewrite the forward kinematics equations (found in lab 1) expliciting the angles.

$$\underline{P} = \begin{pmatrix} x \\ z \end{pmatrix} = \begin{pmatrix} l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2) + l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) \end{pmatrix}$$

As you can immedietaly see, we need to find three angles $\vartheta_{1}$, $\vartheta_{2}$, $\vartheta_{3}$ having only two equations. This means that the problem is **kinematically redundant**, i.e., there can be more than one solution, as we previously saw.

This leave us with an **extra degree of freedom**, i.e., even if we lock one link orientation we can still find a solution to the problem. 

Of course, the the desired end effector position must be **not out of reach**, i.e.,  $\|\underline{P}_{\text{des}}\|_2 < \ell_1 + \ell_2 + \ell_3$.

To deal with this extra degree of freedom, we decide to **lock the last link orientation** to a given angle $\phi$, to get rid of the orientation depending on the most angles (to ease our lifes).

**Remember** from the forward kinematics solution we know that $l_3$ orientation depends on its joint angle $\vartheta_3$ and the previous joints angles $\vartheta_2, \vartheta_3$:

$$\underline{P} = \begin{pmatrix} x \\ z \end{pmatrix} = \begin{pmatrix} l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + \mathbf{l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3)} \\ l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2) + \mathbf{l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3)} \end{pmatrix}$$

This lock adds the following equation:
$$\phi = \vartheta_1 + \vartheta_2 + \vartheta_3$$

Let's visualize its effect:

In [7]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Link lengths and fixed parameters
L1, L2, L3 = 1.2, 1.0, 0.6
TARGET_X, TARGET_Z = 1.6, 0.8
MODE = "elbow-down"

limit = L1 + L2 + L3 + 0.3
circle_theta = np.linspace(0, 2 * np.pi, 100)

def solve_ik(x, z, phi_deg, mode="elbow-down"):
    phi = np.deg2rad(phi_deg)
    
    # 1. Wrist position
    xw = x - L3 * np.cos(phi)
    zw = z - L3 * np.sin(phi)
    
    # 2. Law of cosines for theta2
    D = (xw**2 + zw**2 - L1**2 - L2**2) / (2 * L1 * L2)
    
    if abs(D) > 1.0:
        return None, (xw, zw), D
    
    sign = 1.0 if mode == "elbow-down" else -1.0
    theta2 = np.arctan2(sign * np.sqrt(max(0.0, 1.0 - D**2)), D)
    
    # 3. Solve for theta1
    theta1 = np.arctan2(zw, xw) - np.arctan2(L2 * np.sin(theta2), L1 + L2 * np.cos(theta2))
    
    # 4. Solve for theta3
    theta3 = phi - theta1 - theta2
    
    return (theta1, theta2, theta3), (xw, zw), D

# Initialize FigureWidget
fig = go.FigureWidget()

# 0: Static wrist circle (r = L3) around fixed target
fig.add_trace(go.Scatter(
    x=TARGET_X + L3 * np.cos(circle_theta),
    y=TARGET_Z + L3 * np.sin(circle_theta),
    mode="lines",
    line=dict(color="orange", width=1.5, dash="dash"),
    name="Wrist circle (r = l₃)"
))

# 1, 2, 3: Links 1, 2, 3
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#1f77b4", width=6), name="Link 1 (l₁)"))
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#2ca02c", width=6), name="Link 2 (l₂)"))
fig.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#d62728", width=6), name="Link 3 (l₃)"))

# 4: Joints
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(color="black", size=8),
    showlegend=False
))

# 5: Wrist point
fig.add_trace(go.Scatter(
    x=[], y=[], mode="markers",
    marker=dict(color="orange", size=9),
    name="Wrist (x_w, z_w)"
))

# 6: Static target point
fig.add_trace(go.Scatter(
    x=[TARGET_X], y=[TARGET_Z], mode="markers",
    marker=dict(color="red", size=10),
    name="Target (x, z)"
))

# Static 2-link workspace boundary circles
r_outer = L1 + L2
r_inner = abs(L1 - L2)

fig.update_layout(
    width=650,
    height=650,
    xaxis=dict(range=[-limit, limit], title="x", zeroline=True, zerolinecolor="gray", showgrid=True, gridcolor="rgba(128,128,128,0.2)"),
    yaxis=dict(range=[-limit, limit], title="z", scaleanchor="x", scaleratio=1, zeroline=True, zerolinecolor="gray", showgrid=True, gridcolor="rgba(128,128,128,0.2)"),
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(l=40, r=40, t=70, b=40),
    shapes=[
        dict(type="circle", xref="x", yref="y", x0=-r_outer, y0=-r_outer, x1=r_outer, y1=r_outer,
             line=dict(color="lightgray", width=1.5, dash="dot")),
        dict(type="circle", xref="x", yref="y", x0=-r_inner, y0=-r_inner, x1=r_inner, y1=r_inner,
             line=dict(color="lightgray", width=1.5, dash="dot")),
    ]
)

# Only Phi slider remains interactive
phi_deg_w = widgets.FloatSlider(value=24.0, min=-180.0, max=180.0, step=1.0, description="Phi (deg)")

def update_arm(*args):
    phi_deg = phi_deg_w.value
    angles, (xw, zw), D = solve_ik(TARGET_X, TARGET_Z, phi_deg, MODE)
    
    with fig.batch_update():
        fig.data[5].x, fig.data[5].y = [xw], [zw]
        
        if angles is not None:
            t1, t2, t3 = angles
            x0, z0 = 0.0, 0.0
            x1, z1 = L1 * np.cos(t1), L1 * np.sin(t1)
            x2, z2 = x1 + L2 * np.cos(t1 + t2), z1 + L2 * np.sin(t1 + t2)
            x3, z3 = x2 + L3 * np.cos(t1 + t2 + t3), z2 + L3 * np.sin(t1 + t2 + t3)
            
            fig.data[1].x, fig.data[1].y = [x0, x1], [z0, z1]
            fig.data[2].x, fig.data[2].y = [x1, x2], [z1, z2]
            fig.data[3].x, fig.data[3].y = [x2, x3], [z2, z3]
            fig.data[3].line = dict(color="#d62728", width=6, dash="solid")
            fig.data[4].x, fig.data[4].y = [x0, x1, x2], [z0, z1, z2]
            
            fig.layout.title = dict(
                text=(
                    f"Wrist: ({xw:.2f}, {zw:.2f})   |   D = {D:.3f}<br>"
                    f"θ1 = {np.rad2deg(t1):.1f}°,  θ2 = {np.rad2deg(t2):.1f}°,  θ3 = {np.rad2deg(t3):.1f}°"
                ),
                font=dict(color="black", size=14)
            )
        else:
            fig.data[1].x, fig.data[1].y = [], []
            fig.data[2].x, fig.data[2].y = [], []
            fig.data[4].x, fig.data[4].y = [], []
            fig.data[3].x, fig.data[3].y = [xw, TARGET_X], [zw, TARGET_Z]
            fig.data[3].line = dict(color="#d62728", width=3, dash="dash")
            
            fig.layout.title = dict(
                text=f"Wrist out of reach (|D| = {abs(D):.3f} > 1.0)",
                font=dict(color="red", size=14)
            )

phi_deg_w.observe(update_arm, names="value")
update_arm()

display(widgets.VBox([phi_deg_w, fig]))

Now we have one equation per unknown quantity:

$$\begin{pmatrix} x \\ z \\ \phi \end{pmatrix} = \begin{pmatrix} l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2) + l_3 \cos(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2) + l_3 \sin(\vartheta_1 + \vartheta_2 + \vartheta_3) \\ \vartheta_1 + \vartheta_2 + \vartheta_3\end{pmatrix}$$

And three variables: $x, z, \phi$

### Deliverable 3.1

**Deliverable 3.1:** Find $\vartheta_1, \vartheta_2, \vartheta_3$ starting from $x, z, \phi$

### Solution

Substitute $\phi$ into the $x$ and $z$ equations and move the third link terms ($l_3$) to the left-hand side to isolate the coordinates of the wrist center $(x_w, z_w)$:

$$x_w = x - l_3 \cos(\phi) = l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2)$$

$$z_w = z - l_3 \sin(\phi) = l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2)$$

This reduces the problem to a standard 2-link planar arm reaching the point $(x_w, z_w)$.

Square both wrist equations and add them together to eliminate $\vartheta_1$:

$$x_w^2 + z_w^2 = \left(l_1 \cos(\vartheta_1) + l_2 \cos(\vartheta_1 + \vartheta_2)\right)^2 + \left(l_1 \sin(\vartheta_1) + l_2 \sin(\vartheta_1 + \vartheta_2)\right)^2$$

Expand the right-hand side:

$$x_w^2 + z_w^2 = l_1^2(\cos^2\vartheta_1 + \sin^2\vartheta_1) + l_2^2(\cos^2(\vartheta_1+\vartheta_2) + \sin^2(\vartheta_1+\vartheta_2)) + 2l_1 l_2\left(\cos\vartheta_1\cos(\vartheta_1+\vartheta_2) + \sin\vartheta_1\sin(\vartheta_1+\vartheta_2)\right)$$

Apply the Pythagorean identity ($\cos^2\alpha + \sin^2\alpha = 1$) and the cosine difference identity ($\cos(A - B) = \cos A \cos B + \sin A \sin B$):

$$x_w^2 + z_w^2 = l_1^2 + l_2^2 + 2l_1 l_2 \cos\vartheta_2$$

Solve for $\cos(\vartheta_2)$:

$$\cos(\vartheta_2) = \frac{x_w^2 + z_w^2 - l_1^2 - l_2^2}{2 l_1 l_2} \equiv D$$

Note: A real solution exists only if $\vert{}D\vert{} \le 1$, which means the wrist target is within the arm's reach.

Find $\sin(\vartheta_2)$ using the pythagorean trigonometric identity: $\sin(\vartheta_2) = \pm\sqrt{1 - D^2}$. The two signs correspond to the elbow-up ($-$) and elbow-down ($+$) configurations. Using the two-argument arctangent ($\operatorname{atan2}$) preserves the quadrant:$$\vartheta_2 = \operatorname{atan2}\left(\pm\sqrt{1 - D^2},\; D\right)$$

Expand the $\cos(\vartheta_1 + \vartheta_2)$ and $\sin(\vartheta_1 + \vartheta_2)$ terms in the wrist equations:

$$x_w = (l_1 + l_2 \cos\vartheta_2)\cos\vartheta_1 - (l_2 \sin\vartheta_2)\sin\vartheta_1$$$$z_w = (l_2 \sin\vartheta_2)\cos\vartheta_1 + (l_1 + l_2 \cos\vartheta_2)\sin\vartheta_1$$

Let $k_1 = l_1 + l_2 \cos\vartheta_2$ and $k_2 = l_2 \sin\vartheta_2$. The system becomes a linear rotation:

$$\begin{pmatrix} x_w \\ z_w \end{pmatrix} = \begin{pmatrix} k_1 & -k_2 \\ k_2 & k_1 \end{pmatrix} \begin{pmatrix} \cos\vartheta_1 \\ \sin\vartheta_1 \end{pmatrix}$$

Notice that $\vartheta_1$ is the angle of the vector $(x_w, z_w)$ minus the internal angle offset formed by $(k_1, k_2)$:

$$\vartheta_1 = \operatorname{atan2}(z_w, x_w) - \operatorname{atan2}(k_2, k_1)$$

Substituting $k_1$ and $k_2$ back in:

$$\vartheta_1 = \operatorname{atan2}(z - l_3 \sin\phi,\; x - l_3 \cos\phi) - \operatorname{atan2}(l_2 \sin\vartheta_2,\; l_1 + l_2 \cos\vartheta_2)$$

Now that $\vartheta_1$ and $\vartheta_2$ are known, solve the third original equation directly:

$$\vartheta_3 = \phi - \vartheta_1 - \vartheta_2$$

Summarizing we have:

$$\begin{aligned} \vartheta_2 &= \operatorname{atan2}\!\left(\pm\sqrt{1 - \left(\frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right)^2},\; \frac{(x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 - l_1^2 - l_2^2}{2l_1 l_2}\right) \\ \vartheta_1 &= \operatorname{atan2}(z - l_3\sin\phi,\; x - l_3\cos\phi) - \operatorname{atan2}(l_2 \sin\vartheta_2,\; l_1 + l_2 \cos\vartheta_2) \\ \vartheta_3 &= \phi - \vartheta_1 - \vartheta_2 \end{aligned}$$

Note that we have 3 input: $x, z, \phi$. It's our duty to choose the right end effector orientation to be sure that we can reach the target.

To be sure that the target is reachable the following must be true:

$$\vert{}l_1 - l_2\vert{} \le \sqrt{x_w^2 + z_w^2} \le l_1 + l_2$$

Substituting $x_w = x - l_3\cos\phi$ and $z_w = z - l_3\sin\phi$, and squaring all sides gives the reachability condition in terms of $\phi$:

$$(l_1 - l_2)^2 \le (x - l_3\cos\phi)^2 + (z - l_3\sin\phi)^2 \le (l_1 + l_2)^2$$

Expanding the middle term using $\cos^2\phi + \sin^2\phi = 1$:

$$x^2 + z^2 + l_3^2 - 2l_3(x\cos\phi + z\sin\phi) = r^2 + l_3^2 - 2l_3 r \cos(\phi - \alpha)$$

where $r = \sqrt{x^2 + z^2}$ is the target distance from the base and $\alpha = \operatorname{atan2}(z, x)$ is the target's polar angle. 

Substituting this back into the workspace inequality isolates $\cos(\phi - \alpha)$:

$$\frac{r^2 + l_3^2 - (l_1 + l_2)^2}{2l_3 r} \le \cos(\phi - \alpha) \le \frac{r^2 + l_3^2 - (l_1 - l_2)^2}{2l_3 r}$$

(Note: If the target is beyond the maximum total reach $r > l_1 + l_2 + l_3$, the left bound exceeds $1$ and no $\phi$ can reach the target.)

## Practical Strategies for Selecting $\phi$:

Depending on your application, you can pick $\phi$ from the valid interval above using one of three standard strategies:

- Task-Constrained Orientation: If the task requires a specific approach angle (e.g., $\phi = 0$ for horizontal insertion or $\phi = -\frac{\pi}{2}$ for vertical pick-and-place), check whether $\vert \cos(\vartheta_2) \vert \le 1$. If not, the target cannot be reached at that fixed angle.
- Radial Alignment (Maximum Reach Heuristic): When the task does not constrain the approach angle, the safest choice to guarantee reachability is pointing the third link directly along the vector from the base to the target:$$\phi = \operatorname{atan2}(z, x)$$This maximizes $\cos(\phi - \alpha) = 1$, pulling the wrist center as close to the base as possible ($r_w = \vert{}r - l_3\vert{}$) and maximizing the chance that $\vert \cos(\vartheta_2) \vert \le 1$.
- Interval Search / Nearest-Valid Angle: If a preferred angle $\phi_{\text{pref}}$ violates $\vert \cos(\vartheta_2) \vert \le 1$, clamp $\cos(\phi - \alpha)$ to the nearest boundary of the inequality above and solve for $\phi = \alpha \pm \arccos(\dots)$ to find the closest achievable orientation to $\phi_{\text{pref}}$.

## 2D IK Simulation

Let's simulate it:

In [16]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Link lengths
l1, l2, l3 = 2.0, 2.0, 1.0

def is_reachable(x, z, phi):
    xw = x - l3 * np.cos(phi)
    zw = z - l3 * np.sin(phi)
    D = (xw**2 + zw**2 - l1**2 - l2**2) / (2 * l1 * l2)
    return abs(D) <= 1.0

def auto_compute_phi(x, z):
    # 1. Preferred heuristic: point radially outward from the base
    preferred_phi = np.arctan2(z, x)
    if is_reachable(x, z, preferred_phi):
        return np.rad2deg(preferred_phi)
    
    # 2. Fallback: search angles closest to the preferred radial direction
    offsets = np.linspace(0, np.pi, 180)
    for offset in offsets:
        for sign in (1.0, -1.0):
            candidate = preferred_phi + sign * offset
            if is_reachable(x, z, candidate):
                wrapped = (np.rad2deg(candidate) + 180) % 360 - 180
                return wrapped
                
    return np.rad2deg(preferred_phi)

def solve_ik(x, z, phi_deg, elbow_up=False):
    phi = np.deg2rad(phi_deg)
    
    # 1. Wrist position
    xw = x - l3 * np.cos(phi)
    zw = z - l3 * np.sin(phi)
    
    # 2. Solve for theta_2
    D = (xw**2 + zw**2 - l1**2 - l2**2) / (2 * l1 * l2)
    
    if abs(D) > 1.0:
        return None  # Target out of reach
        
    sign = -1.0 if elbow_up else 1.0
    theta2 = np.arctan2(sign * np.sqrt(1 - D**2), D)
    
    # 3. Solve for theta_1
    theta1 = np.arctan2(zw, xw) - np.arctan2(l2 * np.sin(theta2), l1 + l2 * np.cos(theta2))
    
    # 4. Solve for theta_3
    theta3 = phi - theta1 - theta2
    
    return theta1, theta2, theta3

def forward_kinematics(theta1, theta2, theta3):
    x0, z0 = 0.0, 0.0
    x1 = l1 * np.cos(theta1)
    z1 = l1 * np.sin(theta1)
    x2 = x1 + l2 * np.cos(theta1 + theta2)
    z2 = z1 + l2 * np.sin(theta1 + theta2)
    x3 = x2 + l3 * np.cos(theta1 + theta2 + theta3)
    z3 = z2 + l3 * np.sin(theta1 + theta2 + theta3)
    return [x0, x1, x2, x3], [z0, z1, z2, z3]

# Create FigureWidget
fig = go.FigureWidget()

max_reach = l1 + l2 + l3
circle_theta = np.linspace(0, 2 * np.pi, 200)

# Trace 0: Max Reach Circle
fig.add_trace(go.Scatter(
    x=max_reach * np.cos(circle_theta),
    y=max_reach * np.sin(circle_theta),
    mode='lines',
    line=dict(color='lightgray', dash='dash'),
    name='Max Reach'
))

# Trace 1: Radial Reference Ray (Base -> Target -> Max Reach)
fig.add_trace(go.Scatter(
    x=[0, 0],
    y=[0, 0],
    mode='lines',
    line=dict(color='#2ca02c', width=2, dash='dot'),
    name='Ideal Radial Axis'
))

# Trace 2: End-Effector Heading Indicator
fig.add_trace(go.Scatter(
    x=[0, 0],
    y=[0, 0],
    mode='lines',
    line=dict(color='#ff7f0e', width=3),
    name='Actual φ Heading'
))

# Trace 3: 3R Arm Links
fig.add_trace(go.Scatter(
    x=[0, 0, 0, 0],
    y=[0, 0, 0, 0],
    mode='lines+markers',
    line=dict(color='#1f77b4', width=6),
    marker=dict(size=12, color=['black', '#1f77b4', '#1f77b4', '#ff7f0e']),
    name='3R Arm'
))

# Trace 4: Target Marker
fig.add_trace(go.Scatter(
    x=[3.0],
    y=[2.0],
    mode='markers',
    marker=dict(size=14, symbol='x', color='red'),
    name='Target'
))

fig.update_layout(
    title='3-Link Planar Arm Inverse Kinematics (Radial Alignment)',
    xaxis=dict(range=[-5.5, 5.5], title='X', scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[-5.5, 5.5], title='Z'),
    width=700,
    height=650,
    template='plotly_white'
)

# Interactive controls
x_slider = widgets.FloatSlider(value=3.0, min=-5.0, max=5.0, step=0.1, description='Target X:')
z_slider = widgets.FloatSlider(value=2.0, min=-5.0, max=5.0, step=0.1, description='Target Z:')
auto_phi_checkbox = widgets.Checkbox(value=True, description='Auto-compute φ (Radial Align.)')
phi_slider = widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=1.0, description='Angle φ (°):', disabled=True)
branch_toggle = widgets.ToggleButtons(options=[('Elbow Down (+)', False), ('Elbow Up (-)', True)], description='Branch:')
status_label = widgets.HTML()

def update_plot(change=None):
    x, z, elbow_up = x_slider.value, z_slider.value, branch_toggle.value
    
    # Ideal radial angle from base (0,0) to target (x,z)
    radial_rad = np.arctan2(z, x)
    radial_deg = np.rad2deg(radial_rad)
    
    if auto_phi_checkbox.value:
        phi_slider.disabled = True
        phi = auto_compute_phi(x, z)
        phi_slider.value = round(phi, 1)
    else:
        phi_slider.disabled = False
        phi = phi_slider.value

    phi_rad = np.deg2rad(phi)
    solution = solve_ik(x, z, phi, elbow_up)
    
    # Angular alignment error (wrapped to [-180, 180])
    align_err = (phi - radial_deg + 180) % 360 - 180
    
    with fig.batch_update():
        # Update Target
        fig.data[4].x = [x]
        fig.data[4].y = [z]
        
        # Update Ideal Radial Ray from origin through target to max_reach
        if np.hypot(x, z) > 1e-6:
            fig.data[1].x = [0.0, max_reach * np.cos(radial_rad)]
            fig.data[1].y = [0.0, max_reach * np.sin(radial_rad)]
        else:
            fig.data[1].x = [0.0, 0.0]
            fig.data[1].y = [0.0, 0.0]
            
        # Update Actual Heading Vector extending 0.8 units past the target
        pointer_len = 0.8
        fig.data[2].x = [x, x + pointer_len * np.cos(phi_rad)]
        fig.data[2].y = [z, z + pointer_len * np.sin(phi_rad)]
        
        if solution is None:
            fig.data[3].line.color = 'rgba(255, 0, 0, 0.3)'
            status_label.value = "<b style='color:red;'>Target Out of Reach</b>"
        else:
            t1, t2, t3 = solution
            xs, zs = forward_kinematics(t1, t2, t3)
            fig.data[3].x = xs
            fig.data[3].y = zs
            fig.data[3].line.color = '#1f77b4'
            
            align_badge = (
                "<span style='color:green;'>Aligned (0.0°)</span>"
                if abs(align_err) < 0.1
                else f"<span style='color:#d95f02;'>Offset: {align_err:+.1f}°</span>"
            )
            
            status_label.value = (
                f"<b>Radial Target:</b> {radial_deg:.1f}° &nbsp;|&nbsp; "
                f"<b>Actual φ:</b> {phi:.1f}° ({align_badge})<br>"
                f"<b>Joints:</b> &vartheta;<sub>1</sub> = {np.rad2deg(t1):.1f}°, "
                f"&vartheta;<sub>2</sub> = {np.rad2deg(t2):.1f}°, "
                f"&vartheta;<sub>3</sub> = {np.rad2deg(t3):.1f}°"
            )

for widget in (x_slider, z_slider, phi_slider, branch_toggle, auto_phi_checkbox):
    widget.observe(update_plot, names='value')

update_plot()
controls = widgets.VBox([
    widgets.HBox([x_slider, z_slider]),
    widgets.HBox([auto_phi_checkbox, phi_slider]),
    widgets.HBox([branch_toggle]),
    status_label
])
display(controls, fig)

FigureWidget({
    'data': [{'line': {'color': 'lightgray', 'dash': 'dash'},
              'mode': 'lines',
              'name': 'Max Reach',
              'type': 'scatter',
              'uid': '8b7f14b2-5037-45a2-8fcc-cd6869cd2afa',
              'x': {'bdata': ('AAAAAAAAFEBd8A+5cv0TQEja9IrL9R' ... 'rL9RNAXfAPuXL9E0AAAAAAAAAUQA=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAAAAASFg7yMDTEP5MJWgKdMd' ... 'KdMdS/3xUO8jA0xL8Jc9m/rw/WvA=='),
                    'dtype': 'f8'}},
             {'line': {'color': '#2ca02c', 'dash': 'dot', 'width': 2},
              'mode': 'lines',
              'name': 'Ideal Radial Axis',
              'type': 'scatter',
              'uid': '94ca0197-b22b-474c-9dcb-57eef348135a',
              'x': [0.0, 4.160251471689219],
              'y': [0.0, 2.773500981126145]},
             {'line': {'color': '#ff7f0e', 'width': 3},
              'mode': 'lines',
              'name': 'Actual φ Heading',
              'type': 'scat

## Numerical Methods

As you just experienced getting the analytical solution of IK is a long an error prone process, which not always has a solution. 

When no closed form exists, or for practical convenience, numerical methods are used. These include Jacobian pseudoinverse, damped least squares (Levenberg–Marquardt), Newton–Raphson, and optimization-based solvers such as KDL and TRAC-IK.

## Jacobian-based IK

Theory

## MCU Implementation

Adapt Pinocchio + Casadi workflow from lab 1

## Deployment on MCU

Integration with existing code from lab 2 (safety governor)